# 02 — Data Cleaning & Preprocessing

## Objective

Transform the raw dataset into analysis-ready datasets while preserving important business information such as returns.

Cleaning decisions are explicitly documented rather than silently deleting problematic records.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.utils import (
    get_raw_data_dir,
    get_processed_data_dir
)

from src.cleaning import (
    clean_retail_data,
    identify_returns,
    identify_valid_sales
)

from src.preprocessing import (
    create_outlier_flags
)

## Load Data

In [2]:
DATA_PATH = (
    get_raw_data_dir()
    / "online_retail_II.xlsx"
)

df = pd.read_excel(DATA_PATH)

print(df.shape)

(525461, 8)


## Run Cleaning Pipeline

In [3]:
cleaned = clean_retail_data(df)

print(
    f"Raw rows: {len(df):,}"
)

print(
    f"Cleaned rows: {len(cleaned):,}"
)

Raw rows: 525,461
Cleaned rows: 518,596


## Create Returns Dataset

In [4]:
returns = identify_returns(cleaned)

print(
    f"Return rows: {len(returns):,}"
)

Return rows: 12,302


## Create Sales Dataset

In [5]:
sales = identify_valid_sales(cleaned)

print(
    f"Valid sales rows: {len(sales):,}"
)

Valid sales rows: 504,731


## Outlier Analysis

In [6]:
cleaned = create_outlier_flags(cleaned)

In [7]:
print(
    "IQR outliers:",
    cleaned["UnitPrice_IQR_Outlier"].sum()
)

print(
    "Z-score outliers:",
    cleaned["UnitPrice_ZScore_Outlier"].sum()
)

IQR outliers: 35078
Z-score outliers: 287


## Compare Outlier Methods

In [8]:
outlier_comparison = pd.DataFrame({
    "Method": [
        "IQR",
        "Z-score"
    ],

    "Outliers": [
        cleaned["UnitPrice_IQR_Outlier"].sum(),
        cleaned["UnitPrice_ZScore_Outlier"].sum()
    ]
})

outlier_comparison

,Method,Outliers
0,IQR,35078
1,Z-score,287


## Save Processed Data

In [9]:
processed_dir = get_processed_data_dir()

processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

cleaned.to_csv(
    processed_dir / "online_retail_cleaned.csv",
    index=False
)

sales.to_csv(
    processed_dir / "online_retail_sales.csv",
    index=False
)

returns.to_csv(
    processed_dir / "online_retail_returns.csv",
    index=False
)

## Document the Cleaning Impact

In [10]:
cleaning_summary = pd.DataFrame({
    "Dataset": [
        "Raw",
        "Cleaned",
        "Sales",
        "Returns"
    ],

    "Rows": [
        len(df),
        len(cleaned),
        len(sales),
        len(returns)
    ]
})

cleaning_summary

,Dataset,Rows
0,Raw,525461
1,Cleaned,518596
2,Sales,504731
3,Returns,12302
